# Introduction

Methodology: Taxonomic Harmonization and Cross-Batch Integration
I actually have done the harmonisation myself, however I realise it has to be documanted on line as you did, I only did color code, I wrote the method but It has to be double checked. 
Taxonomic Standardization and Structural Restructuring
To enable robust cross-batch integration and statistical comparison across multi-year sequencing projects, a standardized taxonomic nomenclature was enforced. Raw taxonomic lineages were systematically cleaned to remove low-confidence annotations beyond the genus level, discarding all species-level designations. Standard nomenclature formats were applied uniformly across all datasets using consistent capitalization and joining multi-word terms with underscores.

To optimize algorithmic sorting and indexing, the conventional hierarchical prefix notation was inverted. Lineages failing to resolve to lower taxonomic ranks were refactored from a prefix-dependent format (e.g., unclassified_FamilyName) to a suffix-dependent format (e.g., FamilyName_unclassified). This restructuring allowed for systematic, alphabetical grouping and matrix sorting strictly by hierarchy, moving sequentially from Kingdom down through Phylum, Class, Order, and Family. Missing phylum-level assignments were subsequently harmonized and remapped to modern phylum designations.

A Priori Baseline Selection and Environmental Filtering
Cross-batch harmonization was guided by a deeply curated, high-confidence baseline taxonomy dataset comprising approximately 800 recognized genera. This baseline dataset, established via rigorous biological curation, was designated as the target structural framework. Given that samples spanned disparate computational workflows, laboratories, and collection timelines, the core microbial communities were expected to mirror the specialized ecological niches of the engineering systems under study.

To neutralize batch effects and eliminate false diversity expansions driven by technical artifacts or laboratory-specific misclassifications, unmapped or newly introduced taxa were managed using a conservative ecological filtering protocol. Unidentified or newly introduced operational taxonomic units (OTUs) were systematically collapsed into known baseline genera under the following restrictive conditions:

The target organism shared a direct, identical Family assignment with an established baseline representative.

The biological and ecological traits of the candidate genus aligned strictly with the specific environmental profiles of closed-loop industrial heating and cooling water infrastructure.

Taxa whose primary metadata tied them exclusively to disparate environmental systems—such as marine, deep-ocean, or petroleum reservoirs—were classified as cross-batch noise and omitted from direct genus-level mapping.

Abundance Thresholding, Merging, and Mass Conservation
When the incoming sequencing matrices introduced distinct taxonomic splits (e.g., separating an established baseline genus into multiple newly resolved variants), data integrity was maintained via abundance pooling. The abundance counts of these newly split taxa were collapsed back into the dominant, high-confidence baseline representative by summing their respective cell values, ensuring total absolute mass conservation within the sample profiles.

To protect the dataset from inflation by minor sequencing artifacts while retaining significant biological signals, unclassified lineages at the family level were preserved as valid analytical placeholders only if their individual relative abundance exceeded an ecological significance threshold of greater than 0.01%. High-abundance unclassified fractions critical to specific matrix profiles (such as the dominant unspecific_Bacteria_meta2 faction, representing 66% of Sample 1 total reads) were strictly locked and retained to prevent mathematical distortion of the remaining relative abundance profiles. All manual fusions, data corrections, and abundance shifts were permanently color-flagged within the master matrix to ensure complete traceability. Orange for the Genus representing the 2 or more genus and pink for the cell which get absorbed by the orange. Lila for the genus likely to become the representing genus. 

                     MIC PROJECT
                        │
          ┌─────────────┴─────────────┐
          │                           │
    Biotot_noncured ~800 GENERA
(preserved, never modified)       NEW 150/300 GENERA
          │                           │
          └─────────────┬─────────────┘
                        │
                    MultiTax
                        │
                        ▼
                    GTDB R232
                        │
                        ▼
                Current taxonomy 800-genus harmonised list
                        │
            ┌───────────┴───────────┐
            │                       │
    Update old reference     Classify new data
            │                       │
            ▼                       ▼
    GID identity preserved       Match / new / unknown

In [ ]:
from pathlib import Path
import pandas as pd
from multitax import GtdbTx

#gtdb = GtdbTx(version="232")
#print(gtdb)

GtdbTx(version='232', source=['https://data.gtdb.ecogenomic.org/releases/release232/232.0/ar53_taxonomy_r232.tsv.gz', 'https://data.gtdb.ecogenomic.org/releases/release232/232.0/bac120_taxonomy_r232.tsv.gz'], datetime=datetime.datetime(2026, 7, 23, 0, 26, 49, 383964))


In [ ]:
# Load  current harmonised 800-genus reference
biotot_path = Path("data/Biotot.xlsx")

biotot = pd.read_excel(biotot_path)

biotot.head()

In [11]:
# the querry will have the Genus and keep the ID column, so to not lose the ID, id already in df
genus_queries = biotot[["Genus", "GID"]].copy()
genus_queries["query"] = "g__" + genus_queries["Genus"].astype(str)
genus_queries.head()

,Genus,GID,query
0,02d06,1.0,g__02d06
1,A17,2.0,g__A17
2,Abiotrophia,3.0,g__Abiotrophia
3,Acetanaerobacterium,4.0,g__Acetanaerobacterium
4,Acetivibrio,5.0,g__Acetivibrio


In [12]:
# data from the GTDB database, to get the taxonomic ranks for each genus
#gtdb = GtdbTx(version="232")
rank_prefix = {"d__": "Kingdom", "p__": "Phylum", "c__": "Class", "o__": "Order", "f__": "Family", "g__": "Genus"}

records = []
for _, r in genus_queries.iterrows():
    row = {"GID": r["GID"], "Genus": r["Genus"], "GTDB_status": None}
    try:
        lineage_nodes = gtdb.lineage(r["query"])
        for node in lineage_nodes:
            for prefix, rankname in rank_prefix.items():
                if node.startswith(prefix):
                    row[rankname] = node[len(prefix):]
        row["GTDB_status"] = "found"
    except Exception:
        row["GTDB_status"] = "not_in_gtdb"
    records.append(row)

table_b = pd.DataFrame(records)
table_b.head()

,GID,Genus,GTDB_status,Kingdom,Phylum,Class,Order,Family
0,1.0,02d06,found,NaN,NaN,NaN,NaN,NaN
1,2.0,A17,found,NaN,NaN,NaN,NaN,NaN
2,3.0,Abiotrophia,found,Bacteria,Bacillota,Bacilli,Lactobacillales,Aerococcaceae
3,4.0,Acetanaerobacterium,found,Bacteria,Bacillota,Clostridia,Oscillospirales,Ruminococcaceae
4,5.0,Acetivibrio,found,Bacteria,Bacillota,Clostridia,Acetivibrionales,Acetivibrionaceae


# Query GTDB R232 with MultiTax

Create a gtdb_taxonomy dataframe.
Query data from the GTDB database, to get the taxonomic ranks for each genus

In [13]:
# Query data from the GTDB database, to get the taxonomic ranks for each genus
def strip_unclassified(genus_name):
    """Acetivibrio_unclassified -> Acetivibrio. Returns None if no suffix to strip."""
    for suffix in ("_unclassified", "_uncultured"):
        if genus_name.endswith(suffix):
            return genus_name[: -len(suffix)]
    return None


rank_prefix = {"d__": "Kingdom", "p__": "Phylum", "c__": "Class", "o__": "Order", "f__": "Family", "g__": "Genus"}

def lookup_lineage(query_genus):
    """Returns a dict of rank->value if a real lineage was found, else None."""
    try:
        lineage_nodes = gtdb.lineage(f"g__{query_genus}")
    except Exception:
        return None
    parsed = {}
    for node in lineage_nodes:
        for prefix, rankname in rank_prefix.items():
            if node.startswith(prefix):
                parsed[rankname] = node[len(prefix):]
    # a real hit must contain at least Phylum -- an empty/near-empty result means no match
    if "Phylum" not in parsed:
        return None
    return parsed


records = []
for _, r in genus_queries.iterrows():
    original_genus = r["Genus"]
    row = {"GID": r["GID"], "Genus": original_genus}

    parsed = lookup_lineage(original_genus)
    match_type = "direct"

    if parsed is None:
        stripped = strip_unclassified(original_genus)
        if stripped:
            parsed = lookup_lineage(stripped)
            match_type = f"resolved_via_base_genus({stripped})" if parsed else "not_in_gtdb"
        else:
            match_type = "not_in_gtdb"

    if parsed:
        row.update(parsed)
        row["Genus"] = original_genus  # keep your original name, e.g. Acetivibrio_unclassified, not overwritten
    row["GTDB_status"] = match_type
    records.append(row)

table_b = pd.DataFrame(records)
table_b.head()

AttributeError: 'float' object has no attribute 'endswith'

In [ ]:
merged = biotot.merge(table_b, on="Genus", how="left", suffixes=("_yours", "_gtdb"))

def make_comment(row):
    if row["GTDB_status"] == "skipped_placeholder":
        return "placeholder/composite name, not checked against GTDB"
    if row["GTDB_status"] == "not_in_gtdb":
        return "genus not found in GTDB (may be NCBI-only, or GTDB uses a different name)"
    diffs = []
    for rank in ["Phylum", "Class", "Order", "Family"]:
        yours = row.get(f"{rank}_yours", row.get(rank))
        gtdb_val = row.get(f"{rank}_gtdb")
        if pd.notna(gtdb_val) and yours != gtdb_val:
            diffs.append(f"{rank}: '{yours}' -> '{gtdb_val}'")
    return "; ".join(diffs) if diffs else "matches GTDB"

merged["Comment"] = merged.apply(make_comment, axis=1)
merged[["Genus", "Comment"]].to_excel("data/Biotot_vs_GTDB_comparison.xlsx", index=False)
merged[merged["Comment"].str.startswith(("Phylum", "Class", "Order", "Family"))]

New data Parcing
Found under revising some files and matching it to the late publication on 2024.

In [ ]:
df = pd.read_csv("data/Linage_Family_from_Taxonomietabelle.csv", header=None)

In [ ]:
# Combine Row 0 (TubeID) and Row 1 (FF... ID) to keep track of both sample identifiers
header_cols = []
for col in range(df.shape[1]):
    val1 = str(df.iloc[0, col]).strip() if pd.notna(df.iloc[0, col]) else ""
    val2 = str(df.iloc[1, col]).strip() if pd.notna(df.iloc[1, col]) else ""
    if val1 and val2:
        header_cols.append(f"{val1}_{val2}")
    elif val1:
        header_cols.append(val1)
    elif val2:
        header_cols.append(val2)
    else:
        header_cols.append(f"Col_{col}")
# Slice the dataframe to drop the first two header rows, then apply new column names
df_clean = df.iloc[2:].copy()
df_clean.columns = header_cols
df_clean.reset_index(drop=True, inplace=True)

In [ ]:
# 3. Split the OTU ID from the Kingdom string in the first column
# "4362609.0 k__Bacteria" -> OTUID: "4362609.0", Kingdom: "k__Bacteria"
split_first_col = df_clean.iloc[:, 0].str.split(" ", n=1, expand=True)
df_clean.insert(0, "OTU_ID", split_first_col[0])
df_clean.iloc[:, 1] = split_first_col[1]  # Overwrite the original column with just the Kingdom string

In [ ]:
# Rename taxonomy columns so they are clean and recognizable
tax_ranks = ["Kingdom", "Phylum", "Class", "Order", "Family"]
for i, rank in enumerate(tax_ranks):
    df_clean.rename(columns={df_clean.columns[i + 1]: rank}, inplace=True)
# removing the prefixes "k__", "p__"
tax_columns = ["Kingdom", "Phylum", "Class", "Order", "Family", "Genus"]

# Strip the prefixes (e.g., "k__", "p__") only from these columns
for col in tax_columns:
    if col in df_clean.columns:
        # This replaces any letter followed by '__' at the start of the string
        df_clean[col] = df_clean[col].astype(str).str.replace(r'^[a-z]__', '', regex=True)

The lineage from Kingdom to Family was on a page that was separated from the Genus, so off course was difficult to match them because the reorganisation of the lineage by the experts has been continued a long the years, so it was just merged.

In [ ]:
df_genus = pd.read_csv('data/lineage_genus_from_Taxonomietabelle.csv')
# merge family df with genus df
df_merged = pd.merge(df_clean, df_genus, on='Family') #, how=
# order the columns Otu, Kingdom, Phylum, Class, Order, Family Genus, rest
df_merged = df_merged[["OTU_ID", "Kingdom", "Phylum", "Class", "Order", "Family", "Genus"] + [col for col in df_merged.columns if col not in ["OTU_ID", "Kingdom", "Phylum", "Class", "Order", "Family", "Genus"]]] 

In [ ]:
# saving the df to a excel file
df_merged.to_excel("data/merged_taxonomy_from_taxonomytabelle.xlsx", index=False)